In [1]:
import random

from qibo import (
    Circuit, 
    gates, 
    hamiltonians,
    symbols,
    set_backend,
)

from tncdr.targets.ansatze import TranspiledAnsatz, HardwareEfficient
from tncdr.evolutors.models import HybridSurrogate

set_backend("qibojit", platform="numba")

nqubits = 20
nlayers = 3
obs_str = "Y" * nqubits

def construct_circuit(nqubits: int, nlayers: int):
    """Conctruct a target quantum circuit."""
    c = Circuit(nqubits)
    for _ in range(nlayers):
        for q in range(nqubits):
            c.add(gates.RY(q=q, theta=random.random()))
            c.add(gates.RZ(q=q, theta=random.random()))
        [c.add(gates.CNOT(q0=q%nqubits, q1=(q+1)%nqubits)) for q in range(nqubits)]
    return c

def initial_state(nqubits: int):
    """Construct a random initial state."""
    c = Circuit(nqubits)
    for q in range(nqubits):
        c.add(gates.RY(q=q, theta=random.random()))
    return c

print("Target circuit:")
circ = construct_circuit(nqubits=nqubits, nlayers=nlayers)
circ.draw()

print("\nInitial state circuit:")
init_psi = initial_state(nqubits=nqubits)
init_psi.draw()

circ = construct_circuit(nqubits=nqubits, nlayers=nlayers)

# Construct a transpiled ansatz given a circuit
# Arguments here are native gates and chip topology 
ans = TranspiledAnsatz(original_circuit=circ)

# Construct the Hybrid Surrogate
hs = HybridSurrogate(ansatz=ans, initial_state=init_psi) # Here one can add an initial state (a circuit)

[Qibo 0.2.20|INFO|2025-07-01 15:26:35]: Using qibojit (numba) backend on /CPU:0


Target circuit:
0 :     ─RY─RZ─o─────────────────────────────────────X─RY─RZ─o──────────────── ...
1 :     ─RY─RZ─X─o───────────────────────────────────|─RY─RZ─X─o────────────── ...
2 :     ─RY─RZ───X─o─────────────────────────────────|─RY─RZ───X─o──────────── ...
3 :     ─RY─RZ─────X─o───────────────────────────────|─RY─RZ─────X─o────────── ...
4 :     ─RY─RZ───────X─o─────────────────────────────|─RY─RZ───────X─o──────── ...
5 :     ─RY─RZ─────────X─o───────────────────────────|─RY─RZ─────────X─o────── ...
6 :     ─RY─RZ───────────X─o─────────────────────────|─RY─RZ───────────X─o──── ...
7 :     ─RY─RZ─────────────X─o───────────────────────|─RY─RZ─────────────X─o── ...
8 :     ─RY─RZ───────────────X─o─────────────────────|─RY─RZ───────────────X─o ...
9 :     ─RY─RZ─────────────────X─o───────────────────|─RY─RZ─────────────────X ...
10:     ─RY─RZ───────────────────X─o─────────────────|─RY─RZ────────────────── ...
11:     ─RY─RZ─────────────────────X─o───────────────|─RY─RZ───────────

In [30]:
res = ans.partitionate_circuit(
    replacement_probability=0.5, replacement_method="closest"
)

print(res)

# Original circuit
circ.draw()

(([(4, <qibo.gates.gates.RZ object at 0x7f5d0b5a68d0>), (10, <qibo.gates.gates.RZ object at 0x7f5d0936ad90>), (17, <qibo.gates.gates.RZ object at 0x7f5d0936a990>), (18, <qibo.gates.gates.RZ object at 0x7f5d0936b250>), (22, <qibo.gates.gates.RZ object at 0x7f5d0936af10>), (26, <qibo.gates.gates.RZ object at 0x7f5d0936b210>), (28, <qibo.gates.gates.RZ object at 0x7f5d0936b650>), (32, <qibo.gates.gates.RZ object at 0x7f5d0936ba90>), (38, <qibo.gates.gates.RZ object at 0x7f5d0b58d990>), (40, <qibo.gates.gates.RZ object at 0x7f5d0b58f590>), (41, <qibo.gates.gates.RZ object at 0x7f5d0b58ce10>), (50, <qibo.gates.gates.RZ object at 0x7f5d0b58c050>), (52, <qibo.gates.gates.RZ object at 0x7f5d0b58ee50>), (63, <qibo.gates.gates.RZ object at 0x7f5d0b74fad0>), (65, <qibo.gates.gates.RZ object at 0x7f5d0b74e7d0>), (66, <qibo.gates.gates.RZ object at 0x7f5d0b74ec10>), (70, <qibo.gates.gates.RZ object at 0x7f5d0b74d290>), (74, <qibo.gates.gates.RZ object at 0x7f5d0b74d690>), (81, <qibo.gates.gates.RZ 

In [31]:
# Transpiled and quasi-cliffordized circuit
res[1].draw()

0 :     ─GPI2─RZ─GPI2─Z─RZ────────o─────────────────────────────────────────── ...
1 :     ─GPI2─RZ─GPI2─Z─RZ─Z─GPI2─Z─Z─GPI2─o────────────────────────────────── ...
2 :     ─GPI2─RZ─GPI2─Z─RZ──────────Z─GPI2─Z─Z─GPI2─o───────────────────────── ...
3 :     ─GPI2─RZ─GPI2─Z─RZ───────────────────Z─GPI2─Z─Z─GPI2─o──────────────── ...
4 :     ─GPI2─RZ─GPI2─Z─RZ────────────────────────────Z─GPI2─Z─Z─GPI2─o─────── ...
5 :     ─GPI2─RZ─GPI2─Z─RZ─────────────────────────────────────Z─GPI2─Z─Z─GPI2 ...
6 :     ─GPI2─RZ─GPI2─Z─RZ──────────────────────────────────────────────Z─GPI2 ...
7 :     ─GPI2─RZ─GPI2─Z─RZ──────────────────────────────────────────────────── ...
8 :     ─GPI2─RZ─GPI2─Z─RZ──────────────────────────────────────────────────── ...
9 :     ─GPI2─RZ─GPI2─Z─RZ──────────────────────────────────────────────────── ...
10:     ─GPI2─RZ─GPI2─Z─RZ──────────────────────────────────────────────────── ...
11:     ─GPI2─RZ─GPI2─Z─RZ──────────────────────────────────────────────────── ...
12: 

In [32]:
# Sample a similar circuit and compute expval
expval, partitions = hs.expectation_from_partition(
    observable=obs_str,
    return_partitions=True,
    replacement_probability=0.7,
)

expval

-0.0008653088016607263

In [33]:
# Construct Hamiltonian.
form = 1
for i, pauli in enumerate(obs_str):
    form *= getattr(symbols, pauli)(i)
ham = hamiltonians.SymbolicHamiltonian(form=form)

In [24]:
ham.expectation((init_psi+partitions["full_circuit"])().state())

np.float64(0.00029910352539020167)

### Construct a target quantum circuit

Target circuit:
0: ─RZ─o───X─RZ─o───X─
1: ─RZ─X─o─|─RZ─X─o─|─
2: ─RZ───X─o─RZ───X─o─

Initial state circuit:
0: ─RY─
1: ─RY─
2: ─RY─


### Transpiled ansatz

In [7]:
ans.circuit.draw()

0:     ─RZ────────o──────────Z─GPI2─Z─Z──GPI2─RZ───o──────────Z─GPI2─Z─Z─GPI2 ...
1:     ─RZ─Z─GPI2─Z─Z─GPI2─o────────|─RZ─Z────GPI2─Z─Z─GPI2─o────────|─────── ...
2:     ─RZ──────────Z─GPI2─Z─Z─GPI2─o─RZ─────────────Z─GPI2─Z─Z─GPI2─o─────── ...

0: ... ─
1: ... ─
2: ... ─


### Hybrid surrogate

In [8]:

hdw_hs = HybridSurrogate(
    ansatz=hdw_ans,
    initial_state=init_psi,
)

In [13]:
expval, partitions = hs.expectation_from_partition(
    observable=obs_str,
    return_partitions=True,
    replacement_probability=0.6,
)

expval

<qibo.gates.gates.RZ object at 0x7f522998e890> (0.41788205833524417,)
<qibo.gates.gates.RZ object at 0x7f522977c290> (0.034508046418373906,)
<qibo.gates.gates.RZ object at 0x7f522977d990> (0.7073553489034012,)
<qibo.gates.gates.RZ object at 0x7f522a1f7fd0> (0.5467568866725483,)
<qibo.gates.gates.RZ object at 0x7f522976c710> (0.965917494495791,)
<qibo.gates.gates.RZ object at 0x7f522976c590> (0.5484299537564308,)
0: ─RZ─
1: ────
2: ────
0: ───────────o──────────Z─GPI2─Z─Z─GPI2─
1: ─RZ─Z─GPI2─Z─Z─GPI2─o────────|────────
2: ─RZ──────────Z─GPI2─Z─Z─GPI2─o────────
0: ────────o──────────Z─GPI2─Z─Z─GPI2─
1: ─Z─GPI2─Z─Z─GPI2─o────────|────────
2: ──────────Z─GPI2─Z─Z─GPI2─o────────


0.8092304992113187

In [111]:
hs = HybridSurrogate(
    ansatz=ans,
    initial_state=init_psi
)

expval, partitions = hs.expectation_from_partition(
    replacement_probability=0.6,
    observable=obs_str,
    return_partitions=True,
)

expval

0.5502722890060072

In [26]:
print(
    ham.expectation((init_psi+circuit)().state())
)
print(
    ham.expectation((init_psi+circ)().state())
)

0.21638583864243272
0.2163858386424341
